In [1]:
from dataset import Dataset
from model import Retriever, Augmenter, Generator
from evaluate import Evaluator
import os

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

In [2]:
# Modify the parameters here to find good prompts.
api_key = os.getenv("API_KEY")
file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "hard"
question_level = "hard"
is_sample = True
n = 100
model_name = "llama3.1-70b"
method = "plain"

note_prompts = {
    "easy": "Important Note: Your output will strictly be Yes or No with no other words.",
    "medium": "Important Note: You output must be strictly, with no extra words, separated by comma, a list of nutrients with high or low before the nutrients among these options: carb, protein, sugar, sodium, cholesterol, \
        saturated_fat, calorie. For example, the output is: high_carb, low_protein, high_sugar.",
    "hard": "Important Note: You output must be a Yes or No followed by strictly a list of nutrients with high or low as prefix among these options: carb, protein, sugar, sodium, cholesterol, \
        saturated fat, calorie. For example, the output is: Yes, because the food is high carb, low protein, high sugar.",
}

method_prompts = {
    "plain": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "KAPPING": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
}

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

In [3]:
data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs)
retrieved_graphs = retriever.retrieve(method=method)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

generator = Generator(api_key, model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
results = evaluator.evaluate(task_level, predictions, answers)
print(results)

Generating Predictions: 100%|██████████| 100/100 [02:49<00:00,  1.70s/it]


{'ROUGE-1': 0.6588, 'ROUGE-2': 0.4668, 'ROUGE-L': 0.6486, 'BERT': 0.9434, 'BLEU': 0.3521}
